In [1]:
# survey.py
import numpy as np
import pandas as pd
from typing import Optional, Tuple, Dict, Any

# ---------- helpers ----------
def _safe_pct(x: float) -> float:
    try:
        return round(100 * float(x), 2)
    except Exception:
        return 0.0

def _has_cols(df: pd.DataFrame, cols) -> bool:
    return set(cols).issubset(df.columns)

def _describe_series(s: pd.Series, name: str, q=(0.1, 0.5, 0.9)) -> Dict[str, Any]:
    s = pd.to_numeric(s, errors="coerce").dropna()
    out = {
        "count": int(s.shape[0]),
        "mean": float(s.mean()) if len(s) else 0.0,
        "std": float(s.std()) if len(s) else 0.0,
    }
    for p in q:
        out[f"p{int(p*100)}"] = float(s.quantile(p)) if len(s) else 0.0
    return {name: out}

def _to_native(o):
    import numpy as np
    import pandas as pd

    if isinstance(o, dict):
        return {k: _to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_to_native(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    # pandas NA / NaT -> None
    try:
        if pd.isna(o):
            return None
    except Exception:
        pass
    return o

# ---------- 1) Quy mô & độ thưa ----------
'''
- Quy mô & độ thưa 
U: số users duy nhất. U = n_unique(user_id). 
I: số items duy nhất (ở đây là tv_show_id != 0). I = n_unique(tv_show_id!=0). 
nnz (“non-zeros”): số dòng tương tác sau lọc (tức số ô ≠0 trong ma trận user–item). 
density: mật độ ma trận, tỉ lệ ô ≠0. density = nnz / (U * I). (Càng nhỏ → càng thưa → cần CF cho sparse data, top-K, v.v.)
'''
def survey_size_sparsity(df: pd.DataFrame) -> Dict[str, Any]:
    dfi = df[df["tv_show_id"] != 0] if "tv_show_id" in df.columns else df
    U = dfi["user_id"].nunique()
    I = dfi["tv_show_id"].nunique() if "tv_show_id" in dfi.columns else 0
    I_id_0 = (dfi["tv_show_id"] == 0).sum()
    nnz = int(dfi.shape[0])
    density = (nnz / (U * I)) if (U > 0 and I > 0) else 0.0
    return {"users": U, "items": I, "items tv_show_id = 0": I_id_0, "interactions": nnz, "density": float(density)}

# ---------- 2) Phân bố theo user / item ----------
'''
- Phân bố theo user/item 
+ unique items per user: số item khác nhau mỗi user đã tương tác. Dùng các thống kê: 
p10: phân vị 10% (10% user có ≤ giá trị này). 
median: trung vị (50% user ≤ giá trị này). 
mean: trung bình.
=> Lý do: đo “độ dày” lịch sử của user → quyết định CF có học được không. 
+interactions per item: số tương tác mỗi item nhận được; 
cũng xem p10/median/mean và:% items ≥ k: tỉ lệ item có ít nhất k tương tác (k=5,10,20).
=> Cho biết “độ ấm” của catalog.
'''
def survey_user_item_distribution(df: pd.DataFrame) -> Dict[str, Any]:
    dfi = df[df["tv_show_id"] != 0]
    # user side
    u_pos = dfi.groupby("user_id")["tv_show_id"].nunique()
    user_stats = {
        "user_unique_items": {
            "p10": float(np.percentile(u_pos, 10)) if len(u_pos) else 0.0,
            "median": float(np.median(u_pos)) if len(u_pos) else 0.0,
            "mean": float(np.mean(u_pos)) if len(u_pos) else 0.0,
            "%users>=5": _safe_pct((u_pos >= 5).mean()) if len(u_pos) else 0.0,
            "%users>=10": _safe_pct((u_pos >= 10).mean()) if len(u_pos) else 0.0,
        }
    }
    # item side
    i_pos = dfi.groupby("tv_show_id")["user_id"].size()
    item_stats = {
        "item_interactions": {
            "p10": float(np.percentile(i_pos, 10)) if len(i_pos) else 0.0,
            "median": float(np.median(i_pos)) if len(i_pos) else 0.0,
            "mean": float(np.mean(i_pos)) if len(i_pos) else 0.0,
            "%items>=5": _safe_pct((i_pos >= 5).mean()) if len(i_pos) else 0.0,
            "%items>=10": _safe_pct((i_pos >= 10).mean()) if len(i_pos) else 0.0,
            "%items>=20": _safe_pct((i_pos >= 20).mean()) if len(i_pos) else 0.0,
        }
    }
    return {**user_stats, **item_stats}

# ---------- 3) k-core (lặp đến hội tụ) ----------
'''
- k-core (k_u, k_i): phần con của dữ liệu nơi mỗi user ≥ k_u items và mỗi item ≥ k_i users. 
=> Dùng để xem “xương sống” mà CF/MF học tốt còn lại bao nhiêu.
'''
def kcore(df: pd.DataFrame, k_u=5, k_i=5) -> pd.DataFrame:
    d = df[df["tv_show_id"] != 0][["user_id","tv_show_id"]].copy()
    changed = True
    while changed and len(d):
        changed = False
        keep_u = d.groupby("user_id")["tv_show_id"].transform("nunique") >= k_u
        keep_i = d.groupby("tv_show_id")["user_id"].transform("size") >= k_i
        d2 = d[keep_u & keep_i]
        if d2.shape[0] != d.shape[0]:
            d = d2
            changed = True
    return d

def survey_kcore(df: pd.DataFrame, k_u=5, k_i=5) -> Dict[str, Any]:
    kc = kcore(df, k_u, k_i)
    return {
        "kcore": f"{k_u}-{k_i}",
        "users": int(kc["user_id"].nunique()),
        "items": int(kc["tv_show_id"].nunique()),
        "interactions": int(kc.shape[0]),
    }

# ---------- 4) Session ----------
'''
-Session 
session_id: phiên xem được gộp theo khoảng cách thời gian (ví dụ gap>45’ → phiên mới). 
session length (#views): số lượt xem trong phiên; thống kê p10/median/mean. 
% sessions ≥ 2: tỉ lệ phiên có ≥2 lượt (có chuỗi để học co-visitation/sequence). 
co-visitation strength: mức độ hai item hay cùng xuất hiện trong một phiên/cửa sổ → dùng làm candidate theo phiên.
'''
def survey_session(df: pd.DataFrame) -> Dict[str, Any]:
    if "session_id" not in df.columns:
        return {"session": "missing session_id"}
    sess_size = df.groupby(["user_id","session_id"]).size()
    out = {
        "session_length": {
            "p10": float(np.percentile(sess_size, 10)) if len(sess_size) else 0.0,
            "median": float(np.median(sess_size)) if len(sess_size) else 0.0,
            "mean": float(np.mean(sess_size)) if len(sess_size) else 0.0,
        },
        "%sessions_len>=2": _safe_pct((sess_size >= 2).mean()) if len(sess_size) else 0.0,
    }
    return out

# ---------- 5) Screen-time & chất lượng ----------
'''
- Screen-time & chất lượng tín hiệu 
screen_time: tỉ lệ thời gian xem / thời lượng chương trình (0–1). 
Xem p25/median/p75 và % ≥ ngưỡng bạn đặt. 
overlap_minutes: số phút chồng lấn thực tế (để lọc/độ tin cậy). 
% overlap < 2m: tỉ lệ lượt xem quá ngắn (có thể là nhiễu). 
% show_duration > 4h: tỉ lệ chương trình bất thường dài (EPG lệch?).
'''
def survey_quality(df: pd.DataFrame) -> Dict[str, Any]:
    out = {}
    if "screen_time" in df.columns:
        st = df["screen_time"].clip(0, 1).dropna()
        out["screen_time"] = {
            "p25": float(st.quantile(0.25)) if len(st) else 0.0,
            "median": float(st.quantile(0.5)) if len(st) else 0.0,
            "p75": float(st.quantile(0.75)) if len(st) else 0.0,
            "%>=0.8": _safe_pct((st >= 0.8).mean()) if len(st) else 0.0,
        }
    if {"overlap_s", "show_duration_s"} <= set(df.columns):
        ovl_min = (pd.to_numeric(df["overlap_s"], errors="coerce") / 60.0).fillna(0)
        show_h  = (pd.to_numeric(df["show_duration_s"], errors="coerce") / 3600.0).fillna(0)
        out["overlap_minutes"] = _describe_series(ovl_min, "ovl_min")["ovl_min"]
        out["%overlap<2m"] = _safe_pct((ovl_min < 2).mean()) if len(ovl_min) else 0.0
        out["%show>4h"]    = _safe_pct((show_h > 4).mean()) if len(show_h) else 0.0
    return out

# ---------- 6) Metadata availability ----------
'''
- Metadata (cho content/knowledge) 
metadata availability %: tỉ lệ không thiếu của các cột như tv_show_title, tv_show_category, tv_show_genre_*, vsetv_id, year_of_production, … 
=> Càng đầy đủ → content/knowledge càng hữu dụng.
'''
# def survey_metadata(df: pd.DataFrame) -> Dict[str, Any]:
#     meta_cols = [
#         "tv_show_title","tv_show_category",
#         "tv_show_genre_1","tv_show_genre_2","tv_show_genre_3",
#         "vsetv_id","year_of_production"
#     ]
#     out = {}
#     for c in meta_cols:
#         out[c] = _safe_pct(df[c].notna().mean()) if c in df.columns else 0.0
#     return {"metadata_availability_%": out}

# ---------- 7) Cold-start (cần train & val) ----------
'''
- Cold-start 
% cold users in val/test: phần trăm user ở val/test không xuất hiện trong train. 
% cold items in val/test: phần trăm item ở val/test không xuất hiện trong train. 
=> Cold cao → cần content/knowledge & session để bù.
'''
def survey_cold_start(train_df: pd.DataFrame, val_df: pd.DataFrame) -> Dict[str, Any]:
    tr_u = set(train_df["user_id"].unique())
    tr_i = set(train_df.query("tv_show_id != 0")["tv_show_id"].unique())
    va_u = set(val_df["user_id"].unique())
    va_i = set(val_df.query("tv_show_id != 0")["tv_show_id"].unique())
    cold_u = _safe_pct(len(va_u - tr_u) / len(va_u)) if len(va_u) else 0.0
    cold_i = _safe_pct(len(va_i - tr_i) / len(va_i)) if len(va_i) else 0.0
    return {"%cold_users_in_val": cold_u, "%cold_items_in_val": cold_i}

# ---------- 8) Popularity skew ----------
'''
- Popularity skew 
Top-N share: phần trăm tương tác rơi vào Top-N item phổ biến nhất (N=50/100/500…). 
Skew cao → cẩn thận tránh đề xuất chỉ toàn “hot”; cân nhắc diversify/regularize.
'''
def survey_popularity_skew(df: pd.DataFrame, Ns=(50, 100, 500, 1000)) -> Dict[str, Any]:
    dfi = df[df["tv_show_id"] != 0]
    cnt = dfi.groupby("tv_show_id")["user_id"].size().sort_values(ascending=False)
    total = float(cnt.sum()) if len(cnt) else 1.0
    out = {}
    for n in Ns:
        share = cnt.head(n).sum() / total if len(cnt) else 0.0
        out[f"Top{n}_share_%"] = _safe_pct(share)
    return out

# ---------- Orchestrator ----------
def full_survey(
    df: pd.DataFrame,
    train_df: Optional[pd.DataFrame] = None,
    val_df: Optional[pd.DataFrame] = None,
    pretty: bool = True,
) -> Dict[str, Any]:
    report = {}
    report["size_sparsity"]   = survey_size_sparsity(df)
    report["user_item"]       = survey_user_item_distribution(df)
    report["kcore_5_5"]       = survey_kcore(df, 5, 5)
    report["session"]         = survey_session(df)
    report["quality"]         = survey_quality(df)
    # report["metadata"]        = survey_metadata(df)
    if train_df is not None and val_df is not None:
        report["cold_start"]  = survey_cold_start(train_df, val_df)
    report["popularity_skew"] = survey_popularity_skew(df)

    report_native = _to_native(report)
    if pretty:
        import json
        print(json.dumps(report_native, indent=2, ensure_ascii=False))
    return report_native

In [2]:
import pandas as pd

# 1) Load dữ liệu
logs_train_df = pd.read_parquet("logs_train.parquet")
logs_val_df   = pd.read_parquet("logs_val.parquet")
metadata_df = pd.read_parquet("metadata.parquet")
metadata_test_df = pd.read_parquet("metadata_test.parquet")


full_survey(logs_train_df, train_df=logs_train_df, val_df=logs_val_df)

{
  "size_sparsity": {
    "users": 4838,
    "items": 3716,
    "items tv_show_id = 0": 0,
    "interactions": 931775,
    "density": 0.05182860081050136
  },
  "user_item": {
    "user_unique_items": {
      "p10": 17.0,
      "median": 66.5,
      "mean": 82.23232740801984,
      "%users>=5": 97.89,
      "%users>=10": 95.23
    },
    "item_interactions": {
      "p10": 5.0,
      "median": 44.0,
      "mean": 250.7467707212056,
      "%items>=5": 90.47,
      "%items>=10": 82.1,
      "%items>=20": 70.4
    }
  },
  "kcore_5_5": {
    "kcore": "5-5",
    "users": 4735,
    "items": 3362,
    "interactions": 930553
  },
  "session": {
    "session_length": {
      "p10": 1.0,
      "median": 1.0,
      "mean": 2.062176922114496
    },
    "%sessions_len>=2": 44.83
  },
  "quality": {
    "screen_time": {
      "p25": 0.5030555555555556,
      "median": 0.8318181818181818,
      "p75": 1.0,
      "%>=0.8": 52.22
    }
  },
  "cold_start": {
    "%cold_users_in_val": 0.61,
    "%cold

{'size_sparsity': {'users': 4838,
  'items': 3716,
  'items tv_show_id = 0': 0,
  'interactions': 931775,
  'density': 0.05182860081050136},
 'user_item': {'user_unique_items': {'p10': 17.0,
   'median': 66.5,
   'mean': 82.23232740801984,
   '%users>=5': 97.89,
   '%users>=10': 95.23},
  'item_interactions': {'p10': 5.0,
   'median': 44.0,
   'mean': 250.7467707212056,
   '%items>=5': 90.47,
   '%items>=10': 82.1,
   '%items>=20': 70.4}},
 'kcore_5_5': {'kcore': '5-5',
  'users': 4735,
  'items': 3362,
  'interactions': 930553},
 'session': {'session_length': {'p10': 1.0,
   'median': 1.0,
   'mean': 2.062176922114496},
  '%sessions_len>=2': 44.83},
 'quality': {'screen_time': {'p25': 0.5030555555555556,
   'median': 0.8318181818181818,
   'p75': 1.0,
   '%>=0.8': 52.22}},
 'cold_start': {'%cold_users_in_val': 0.61, '%cold_items_in_val': 23.35},
 'popularity_skew': {'Top50_share_%': 54.81,
  'Top100_share_%': 63.57,
  'Top500_share_%': 80.32,
  'Top1000_share_%': 89.53}}